In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE
import joblib

In [2]:
trainperf = pd.read_csv('trainperf.csv')
traindemographics = pd.read_csv('traindemographics.csv')
trainprevloans = pd.read_csv('trainprevloans.csv')

print(trainperf.shape)
print(traindemographics.shape)
print(trainprevloans.shape)

(4368, 10)
(4346, 9)
(18183, 12)


In [3]:
prevloan_agg = trainprevloans.groupby('customerid').agg(
    num_previous_loans=('systemloanid', 'count'),
    avg_previous_loanamount=('loanamount', 'mean'),
    max_previous_loanamount=('loanamount', 'max'),
    avg_previous_totaldue=('totaldue', 'mean'),
    avg_previous_termdays=('termdays', 'mean'),
).reset_index()

print(prevloan_agg.shape)

(4359, 6)


In [4]:
df = trainperf.merge(traindemographics, on='customerid', how='left')
df = df.merge(prevloan_agg, on='customerid', how='left')

print(df.shape)

(4376, 23)


In [5]:
df = df.drop(columns=['referredby', 'bank_branch_clients', 'longitude_gps', 'latitude_gps'])

cat_cols = ['bank_account_type', 'bank_name_clients', 
            'employment_status_clients', 'level_of_education_clients']
df[cat_cols] = df[cat_cols].fillna('Unknown')

df['birthdate'] = pd.to_datetime(df['birthdate'])
df['age'] = 2017 - df['birthdate'].dt.year
df['age'] = df['age'].fillna(df['age'].median())
df = df.drop(columns=['birthdate'])

agg_cols = ['num_previous_loans', 'avg_previous_loanamount', 
            'max_previous_loanamount', 'avg_previous_totaldue', 
            'avg_previous_termdays']
df[agg_cols] = df[agg_cols].fillna(0)

print(df.shape)
print(df.isnull().sum())

(4376, 19)
customerid                    0
systemloanid                  0
loannumber                    0
approveddate                  0
creationdate                  0
loanamount                    0
totaldue                      0
termdays                      0
good_bad_flag                 0
bank_account_type             0
bank_name_clients             0
employment_status_clients     0
level_of_education_clients    0
num_previous_loans            0
avg_previous_loanamount       0
max_previous_loanamount       0
avg_previous_totaldue         0
avg_previous_termdays         0
age                           0
dtype: int64


In [6]:
df['interest_rate'] = (df['totaldue'] - df['loanamount']) / df['loanamount']
df['daily_payment'] = df['totaldue'] / df['termdays']
df['is_repeat_borrower'] = (df['num_previous_loans'] > 0).astype(int)
df['target'] = (df['good_bad_flag'] == 'Good').astype(int)

df['loan_growth'] = df['loanamount'] / (df['avg_previous_loanamount'] + 1)
df['debt_ratio'] = df['totaldue'] / (df['loanamount'] + 1)
df['loan_per_day'] = df['loanamount'] / (df['termdays'] + 1)
df['first_time_large_loan'] = ((df['loannumber'] == 1) & (df['loanamount'] > 30000)).astype(int)
df['young_large_loan'] = ((df['age'] < 25) & (df['loanamount'] > 20000)).astype(int)
df['very_young'] = (df['age'] < 22).astype(int)
df['loan_jump'] = ((df['loanamount'] > df['avg_previous_loanamount'] * 2) & (df['avg_previous_loanamount'] > 0)).astype(int)

risky_statuses = ['Contract', 'Student', 'Unemployed', 'Unknown']
df['no_income_flag'] = df['employment_status_clients'].isin(['Student', 'Unemployed', 'Unknown']).astype(int)
df['high_loan_risky_employment'] = ((df['employment_status_clients'].isin(risky_statuses)) & (df['loanamount'] > 30000)).astype(int)

df['low_edu_large_loan'] = ((df['level_of_education_clients'].isin(['Primary', 'Secondary', 'Unknown'])) & (df['loanamount'] > 30000)).astype(int)
df['educated_but_unemployed'] = ((df['level_of_education_clients'].isin(['Graduate', 'Post-Graduate'])) & (df['employment_status_clients'].isin(['Student', 'Unemployed', 'Unknown']))).astype(int)
df['educated_employed'] = ((df['level_of_education_clients'].isin(['Graduate', 'Post-Graduate'])) & (df['employment_status_clients'].isin(['Permanent', 'Self-Employed']))).astype(int)

df['composite_risk'] = (
    df['no_income_flag'] + 
    df['very_young'] + 
    df['high_loan_risky_employment'] +
    df['first_time_large_loan'] +
    df['low_edu_large_loan']
)

print(df.shape)
print("Feature engineering complete")

(4376, 36)
Feature engineering complete


In [7]:
le = LabelEncoder()
encoding_maps = {}

for col in cat_cols:
    le.fit(df[col].astype(str))
    encoding_maps[col] = dict(enumerate(le.classes_))
    df[col] = le.transform(df[col].astype(str))
    print(f"\n{col}:")
    for i, label in encoding_maps[col].items():
        print(f"  {i} = {label}")


bank_account_type:
  0 = Current
  1 = Other
  2 = Savings
  3 = Unknown

bank_name_clients:
  0 = Access Bank
  1 = Diamond Bank
  2 = EcoBank
  3 = FCMB
  4 = Fidelity Bank
  5 = First Bank
  6 = GT Bank
  7 = Heritage Bank
  8 = Keystone Bank
  9 = Skye Bank
  10 = Stanbic IBTC
  11 = Standard Chartered
  12 = Sterling Bank
  13 = UBA
  14 = Union Bank
  15 = Unity Bank
  16 = Unknown
  17 = Wema Bank
  18 = Zenith Bank

employment_status_clients:
  0 = Contract
  1 = Permanent
  2 = Retired
  3 = Self-Employed
  4 = Student
  5 = Unemployed
  6 = Unknown

level_of_education_clients:
  0 = Graduate
  1 = Post-Graduate
  2 = Primary
  3 = Secondary
  4 = Unknown


In [8]:
drop_cols = ['customerid', 'systemloanid', 'approveddate', 'creationdate', 'good_bad_flag']
df_model = df.drop(columns=drop_cols)

print(df_model.shape)
print(df_model.columns.tolist())

(4376, 31)
['loannumber', 'loanamount', 'totaldue', 'termdays', 'bank_account_type', 'bank_name_clients', 'employment_status_clients', 'level_of_education_clients', 'num_previous_loans', 'avg_previous_loanamount', 'max_previous_loanamount', 'avg_previous_totaldue', 'avg_previous_termdays', 'age', 'interest_rate', 'daily_payment', 'is_repeat_borrower', 'target', 'loan_growth', 'debt_ratio', 'loan_per_day', 'first_time_large_loan', 'young_large_loan', 'very_young', 'loan_jump', 'no_income_flag', 'high_loan_risky_employment', 'low_edu_large_loan', 'educated_but_unemployed', 'educated_employed', 'composite_risk']


In [9]:
X = df_model.drop(columns=['target'])
y = df_model['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", y_train_resampled.value_counts().to_dict())

Before SMOTE: {1: 2738, 0: 762}
After SMOTE: {1: 2738, 0: 2738}


In [10]:
lr = LogisticRegression(max_iter=5000, random_state=42)
lr.fit(X_train_resampled, y_train_resampled)
print("=== Logistic Regression ===")
print("ROC-AUC:", roc_auc_score(y_test, lr.predict_proba(X_test)[:,1]))

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_resampled, y_train_resampled)
print("=== Random Forest ===")
print("ROC-AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]))

xgb = XGBClassifier(random_state=42, eval_metric='logloss')
xgb.fit(X_train_resampled, y_train_resampled)
print("=== XGBoost ===")
print("ROC-AUC:", roc_auc_score(y_test, xgb.predict_proba(X_test)[:,1]))

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


=== Logistic Regression ===
ROC-AUC: 0.6002292964420834
=== Random Forest ===
ROC-AUC: 0.6008369320136049
=== XGBoost ===
ROC-AUC: 0.6453701226735964


In [11]:
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_leaf': [1, 2, 5]
}

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), 
                        rf_params, cv=5, scoring='roc_auc', n_jobs=-1)
rf_grid.fit(X_train_resampled, y_train_resampled)

best_rf = rf_grid.best_estimator_
rf_preds = best_rf.predict(X_test)
rf_proba = best_rf.predict_proba(X_test)[:,1]

print("Best RF params:", rf_grid.best_params_)
print(classification_report(y_test, rf_preds))
print("RF Test ROC-AUC:", roc_auc_score(y_test, rf_proba))

Best RF params: {'max_depth': 15, 'min_samples_leaf': 1, 'n_estimators': 300}
              precision    recall  f1-score   support

           0       0.33      0.41      0.36       191
           1       0.82      0.76      0.79       685

    accuracy                           0.69       876
   macro avg       0.57      0.59      0.58       876
weighted avg       0.71      0.69      0.70       876

RF Test ROC-AUC: 0.6497955440058089


In [12]:
xgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2]
}

xgb_grid = GridSearchCV(XGBClassifier(random_state=42, eval_metric='logloss'), 
                        xgb_params, cv=5, scoring='roc_auc', n_jobs=-1)
xgb_grid.fit(X_train_resampled, y_train_resampled)

best_xgb = xgb_grid.best_estimator_
xgb_preds = best_xgb.predict(X_test)
xgb_proba = best_xgb.predict_proba(X_test)[:,1]

print("Best XGB params:", xgb_grid.best_params_)
print(classification_report(y_test, xgb_preds))
print("XGB Test ROC-AUC:", roc_auc_score(y_test, xgb_proba))

Best XGB params: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200}
              precision    recall  f1-score   support

           0       0.36      0.30      0.33       191
           1       0.81      0.85      0.83       685

    accuracy                           0.73       876
   macro avg       0.59      0.58      0.58       876
weighted avg       0.71      0.73      0.72       876

XGB Test ROC-AUC: 0.6352390415408721


In [13]:
best_rf_final = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=1,
    random_state=42
)
best_rf_final.fit(X_train_resampled, y_train_resampled)

final_preds = best_rf_final.predict(X_test)
final_proba = best_rf_final.predict_proba(X_test)[:,1]

print(classification_report(y_test, final_preds))
print("Final ROC-AUC:", roc_auc_score(y_test, final_proba))

joblib.dump(best_rf_final, 'loan_default_model.pkl')
print("Model saved!")

print("\nFinal feature order:")
print(X.columns.tolist())

              precision    recall  f1-score   support

           0       0.33      0.41      0.36       191
           1       0.82      0.76      0.79       685

    accuracy                           0.69       876
   macro avg       0.57      0.59      0.58       876
weighted avg       0.71      0.69      0.70       876

Final ROC-AUC: 0.6497955440058089
Model saved!

Final feature order:
['loannumber', 'loanamount', 'totaldue', 'termdays', 'bank_account_type', 'bank_name_clients', 'employment_status_clients', 'level_of_education_clients', 'num_previous_loans', 'avg_previous_loanamount', 'max_previous_loanamount', 'avg_previous_totaldue', 'avg_previous_termdays', 'age', 'interest_rate', 'daily_payment', 'is_repeat_borrower', 'loan_growth', 'debt_ratio', 'loan_per_day', 'first_time_large_loan', 'young_large_loan', 'very_young', 'loan_jump', 'no_income_flag', 'high_loan_risky_employment', 'low_edu_large_loan', 'educated_but_unemployed', 'educated_employed', 'composite_risk']


In [15]:
import pandas as pd

test_profiles = pd.DataFrame([
    # Strong profile: Permanent, GT Bank, Graduate, repeat borrower
    {'loannumber': 2, 'loanamount': 10000, 'totaldue': 11500, 'termdays': 30,
     'bank_account_type': 2, 'bank_name_clients': 6, 'employment_status_clients': 1,
     'level_of_education_clients': 0, 'num_previous_loans': 1, 'avg_previous_loanamount': 8000,
     'max_previous_loanamount': 8000, 'avg_previous_totaldue': 9200, 'avg_previous_termdays': 30,
     'age': 28, 'interest_rate': 0.15, 'daily_payment': 383.3, 'is_repeat_borrower': 1,
     'loan_growth': 1.25, 'debt_ratio': 1.15, 'loan_per_day': 333.3, 'first_time_large_loan': 0,
     'young_large_loan': 0, 'very_young': 0, 'loan_jump': 0, 'no_income_flag': 0,
     'high_loan_risky_employment': 0, 'low_edu_large_loan': 0, 'educated_but_unemployed': 0,
     'educated_employed': 1, 'composite_risk': 0},

    # Weak profile: Student, ₦50k, no history
    {'loannumber': 1, 'loanamount': 50000, 'totaldue': 57500, 'termdays': 30,
     'bank_account_type': 1, 'bank_name_clients': 16, 'employment_status_clients': 4,
     'level_of_education_clients': 2, 'num_previous_loans': 0, 'avg_previous_loanamount': 0,
     'max_previous_loanamount': 0, 'avg_previous_totaldue': 0, 'avg_previous_termdays': 0,
     'age': 19, 'interest_rate': 0.15, 'daily_payment': 1916.7, 'is_repeat_borrower': 0,
     'loan_growth': 50000, 'debt_ratio': 1.15, 'loan_per_day': 1612.9, 'first_time_large_loan': 1,
     'young_large_loan': 1, 'very_young': 1, 'loan_jump': 0, 'no_income_flag': 1,
     'high_loan_risky_employment': 1, 'low_edu_large_loan': 1, 'educated_but_unemployed': 0,
     'educated_employed': 0, 'composite_risk': 4},
])

# Ensure column order matches training exactly
test_profiles = test_profiles[X.columns.tolist()]

probs = best_rf_final.predict_proba(test_profiles)[:, 1]
for i, p in enumerate(probs):
    print(f"Profile {i+1}: {round(p*100, 2)}% repayment probability")

Profile 1: 79.62% repayment probability
Profile 2: 49.22% repayment probability
